[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/optimization/currency-MV.ipynb)

In [1]:
if 'google.colab' in str(get_ipython()):
    import os
    os.system('pip -qqq install ModelFlowIb')
    os.system('curl -L -o exchangerates_get.py https://raw.githubusercontent.com/IbHansen/wb-debt-simulation/main/optimization/exchangerates_get.py')

In [2]:
import pandas as pd
import numpy as np

import exchangerates_get as er

# Flip to True to force a fresh ECB download and overwrite the cache.
REFRESH = False


In [3]:
fx_eur = er.ecb_fx_eur(
    currencies=["USD", "GBP", "JPY", "CHF","EUR","ZAR"],
    start="2010-01-01",
    freq='Q',
    refresh=REFRESH,
)
#fx_eur    

In [4]:
# Step 2: express everything in base currency 
fx_ccy = er.convert_base_currency(fx_eur, base="zar")

fx_returns = er.get_fx_returns(fx_ccy)
# Step 3: yearly covariance matrices
fx_cov = er.get_fx_covariance(fx_returns)

In [5]:
cov_df = fx_cov.rename(
    index=lambda x: x.split('_')[1],
    columns=lambda x: x.split('_')[1],
)
names = cov_df.index

assumptions = pd.DataFrame(
    {
        'interest_rate':         [0.010, 0.020, 0.023, 0.013, 0.034],
        'expected_appreciation': [0.000, 0.000, 0.000, 0.000, 0.000],
        'min_share':             [0.0, 0.0, 0.0, 0.0, 0.0],
        'max_share':             [1.0, 1.0, 1.0, 1.0, 1.0],
        'current_share':         [0.2, 0.2, 0.2, 0.2, 0.2],
    },
    index=names,
)



## Interactive inputs — `DebtFrontierInputs`

The cells below wrap the same covariance matrix and assumption frame in a `DebtFrontierInputs` dataclass and render an editable grid (currencies × input fields). Edit any cell, then press **Run frontier** to re-solve and re-plot. The dataclass's `assumptions` attribute always reflects the latest edits, so you can also drive `inputs.plot()` from code between clicks.

In [6]:
inputs = er.DebtFrontierInputs(cov_df=cov_df, assumptions=assumptions)
inputs.assumptions

,interest_rate,expected_appreciation,min_share,max_share,current_share
CHF,0.010,0.0,0.0,1.0,0.2
GBP,0.020,0.0,0.0,1.0,0.2
JPY,0.023,0.0,0.0,1.0,0.2
USD,0.013,0.0,0.0,1.0,0.2
EUR,0.034,0.0,0.0,1.0,0.2


In [7]:
inputs.widget()

## Additional frontier — including domestic debt

Add a domestic-currency (ZAR) row with **risk = 0** (no FX exposure) and an interest rate of **5%** as a placeholder. Both `cov_df` and `assumptions` are extended; the widget below picks up the new row automatically so you can edit the rate, bounds, or current share.


In [8]:
# Extend cov_df with a ZAR (domestic) row/column of zeros — no FX risk.
cov_df_dom = cov_df.reindex(
    index=list(cov_df.index) + ['ZAR'],
    columns=list(cov_df.columns) + ['ZAR'],
    fill_value=0.0,
)

# Add a ZAR assumptions row.  current_share = 0 means the baseline
# portfolio has no domestic borrowing yet, so the foreign shares still
# sum to 1.0 and the widget's Σ indicator stays green.
zar_row = pd.DataFrame(
    {
        'interest_rate':         [0.05],   # 5% — placeholder
        'expected_appreciation': [0.0],
        'min_share':             [0.0],
        'max_share':             [1.0],
        'current_share':         [0.0],
    },
    index=['ZAR'],
)
assumptions_dom = pd.concat([assumptions, zar_row])

inputs_dom = er.DebtFrontierInputs(
    cov_df=cov_df_dom,
    assumptions=assumptions_dom,
    name='basis_with_domestic',
)
inputs_dom.widget()
